In [1]:
!git clone --filter=blob:none --sparse https://github.com/TheLedAlchemist/ARENA_materials.git

Cloning into 'ARENA_materials'...
remote: Enumerating objects: 16190, done.
remote: Total 16190 (delta 0), reused 0 (delta 0), pack-reused 16190 (from 2)
Receiving objects: 100% (16190/16190), 1.96 MiB | 26.78 MiB/s, done.
Resolving deltas: 100% (10124/10124), done.
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 11 (delta 0), reused 0 (delta 0), pack-reused 5 (from 2)
Receiving objects: 100% (11/11), 9.57 MiB | 19.37 MiB/s, done.
Updating files: 100% (11/11), done.


In [2]:
%cd ARENA_materials
!git sparse-checkout set chapter0_fundamentals
!pip install torchinfo
!pip install jaxtyping

/content/ARENA_materials
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 69 (delta 11), reused 1 (delta 1), pack-reused 25 (from 1)
Receiving objects: 100% (69/69), 1.78 MiB | 14.97 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Updating files: 100% (69/69), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jaxtyping]


In [8]:
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable, Literal

import os
os.chdir("/content/ARENA_materials")

import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as F
import wandb
from IPython.core.display import HTML
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor, optim
from torch.utils.data import DataLoader, DistributedSampler
from tqdm.auto import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part3_optimization"
root_dir = Path("/content/ARENA_materials")
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))



MAIN = __name__ == "__main__"

import part3_optimization.tests as tests
from fundamentals_utils import IMAGENET_TRANSFORM, CIFAR10
from part2_cnns.solutions import Linear, ResNet34, get_resnet_for_feature_extraction
from part3_optimization.utils import plot_fn, plot_fn_with_points
from plotly_utils import bar, imshow, line

# Cap the number of CPU threads PyTorch uses. On machines with many cores the default of one
# thread per core makes small CPU tensor operations much slower
torch.set_num_threads(min(4, torch.get_num_threads()))

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # pick the fastest conv algorithms for our fixed batch shapes (the ResNet fine-tunes)
